# Multi-Seed Robustness of Script-Adversarial Writer Verification

## Research question

Does the writer-verification improvement observed for batch-level alternating script-adversarial training remain consistent when the training random seed changes?

## Experimental comparison

Two configurations are compared:

1. **Lambda-0 control**
   - SSL-initialized dynamic writer metric learning
   - identical writer-verification objective
   - no adversarial gradient reaches the writer encoder

2. **Batch-alternating lambda-0.5**
   - same SSL initialization
   - same writer metric-learning protocol
   - script adversary attached to the 512-D embedding
   - GRL coefficient = 0.5
   - 3 adversary updates per batch

The writer split, verification protocol, dynamic negative-writer offsets, preprocessing, architecture, training budget, and checkpoint-selection rule remain fixed.

Only training stochasticity changes across seeds.

## Seeds

- 42 — existing Notebook 16 result
- 123 — new run
- 2026 — new run

Seed values were fixed before inspecting the new runs.

## Primary evaluation

For each seed:

- validation writer ROC-AUC
- canonical interpolated EER
- cross-script macro ROC-AUC
- independent writer-disjoint script-probe ROC-AUC

The main comparison is paired within each seed:

**batch-alternating lambda-0.5 minus lambda-0 control**

No official-test results will be used for multi-seed model or method selection.

In [1]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

from handwriting_cross_script_research.dataset import QUWIDataset
from handwriting_cross_script_research.preprocessing import load_image_tensor

In [2]:
SPLIT_SEED = 42

TRAINING_SEEDS = [
    42,
    123,
    2026,
]

NEW_TRAINING_SEEDS = [
    123,
    2026,
]

EPOCHS = 10

METRIC_BATCH_SIZE = 8
SCRIPT_BATCH_SIZE = 16

ENCODER_LR = 1e-5
SCRIPT_HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_MAX_NORM = 5.0
COSINE_MARGIN = 0.5

GRL_COEFFICIENT = 0.5
ADVERSARY_STEPS_PER_BATCH = 3

DYNAMIC_NEGATIVE_OFFSETS = [
    222,
    168,
    20,
    143,
    97,
    96,
    156,
    22,
    46,
    190,
]

if (Path.cwd() / "pyproject.toml").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "pyproject.toml").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

SSL_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "self_supervised_pretraining"
    / "resnet18_moco_ssl_best.pt"
)

NOTEBOOK16_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "script_adversarial_metric_learning"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "multiseed_adversarial_robustness"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "multiseed_adversarial_robustness"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Training seeds:", TRAINING_SEEDS)
print("New training seeds:", NEW_TRAINING_SEEDS)
print("Epochs per run:", EPOCHS)

Project root: /home/arijit/Documents/handwriting-cross-script-research
Training seeds: [42, 123, 2026]
New training seeds: [123, 2026]
Epochs per run: 10


In [5]:
split_df = pd.read_csv(
    SPLIT_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

seed42_tradeoff_df = pd.read_csv(
    NOTEBOOK16_REPORT_DIR
    / "validation_utility_leakage_tradeoff.csv"
)

seed42_control_row = (
    seed42_tradeoff_df[
        seed42_tradeoff_df[
            "model"
        ] == "lambda0_control"
    ]
    .iloc[0]
)

seed42_batch_alt_row = (
    seed42_tradeoff_df[
        seed42_tradeoff_df[
            "model"
        ] == "batch_alt_lambda0p5"
    ]
    .iloc[0]
)

print(
    "Development writers:",
    development_df[
        "writer"
    ].nunique(),
)

print(
    "Development images:",
    len(
        development_df
    ),
)

print(
    "Validation writers:",
    validation_df[
        "writer"
    ].nunique(),
)

print(
    "Validation images:",
    len(
        validation_df
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print()

print(
    "Seed-42 control AUC:",
    seed42_control_row[
        "writer_auc"
    ],
)

print(
    "Seed-42 batch-alt AUC:",
    seed42_batch_alt_row[
        "writer_auc"
    ],
)

print(
    "Seed-42 AUC gain:",
    (
        seed42_batch_alt_row[
            "writer_auc"
        ]
        - seed42_control_row[
            "writer_auc"
        ]
    ),
)

assert development_df[
    "writer"
].nunique() == 226

assert len(
    development_df
) == 904

assert validation_df[
    "writer"
].nunique() == 56

assert len(
    validation_df
) == 224

assert len(
    validation_pairs_df
) == 18816

Development writers: 226
Development images: 904
Validation writers: 56
Validation images: 224
Validation pairs: 18816

Seed-42 control AUC: 0.7441469864460937
Seed-42 batch-alt AUC: 0.785573689703154
Seed-42 AUC gain: 0.04142670325706033


In [6]:
def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if torch.cuda.is_available():
    try:
        test_tensor = torch.ones(
            1,
            device="cuda",
        )

        TRAIN_DEVICE = torch.device(
            "cuda"
        )

        del test_tensor
        torch.cuda.empty_cache()

    except RuntimeError:
        TRAIN_DEVICE = torch.device(
            "cpu"
        )
else:
    TRAIN_DEVICE = torch.device(
        "cpu"
    )


set_training_seed(
    SPLIT_SEED
)

print(
    "Training device:",
    TRAIN_DEVICE,
)

print(
    "Initial seed:",
    SPLIT_SEED,
)

Training device: cuda
Initial seed: 42


In [11]:
page1_df = (
    development_df[
        development_df[
            "page_id"
        ] == 1
    ]
    .sort_values(
        "writer"
    )
    .reset_index(drop=True)
)

page2_df = (
    development_df[
        development_df[
            "page_id"
        ] == 2
    ]
    .sort_values(
        "writer"
    )
    .reset_index(drop=True)
)

writers = page1_df[
    "writer"
].to_numpy()

assert np.array_equal(
    writers,
    page2_df[
        "writer"
    ].to_numpy(),
)

page1_filename = dict(
    zip(
        page1_df[
            "writer"
        ],
        page1_df[
            "filename"
        ],
    )
)

page2_filename = dict(
    zip(
        page2_df[
            "writer"
        ],
        page2_df[
            "filename"
        ],
    )
)


def build_dynamic_training_pairs():
    schedule_rows = []

    for epoch, offset in enumerate(
        DYNAMIC_NEGATIVE_OFFSETS,
        start=1,
    ):
        negative_indices = np.roll(
            np.arange(
                len(
                    writers
                )
            ),
            -int(
                offset
            ),
        )

        epoch_rows = []

        for anchor_index, anchor_writer in enumerate(
            writers
        ):
            negative_writer = writers[
                negative_indices[
                    anchor_index
                ]
            ]

            epoch_rows.append(
                {
                    "epoch": epoch,
                    "offset": int(
                        offset
                    ),
                    "anchor_writer": int(
                        anchor_writer
                    ),
                    "partner_writer": int(
                        anchor_writer
                    ),
                    "filename_a": page1_filename[
                        anchor_writer
                    ],
                    "filename_b": page2_filename[
                        anchor_writer
                    ],
                    "pair_label": 1,
                }
            )

            epoch_rows.append(
                {
                    "epoch": epoch,
                    "offset": int(
                        offset
                    ),
                    "anchor_writer": int(
                        anchor_writer
                    ),
                    "partner_writer": int(
                        negative_writer
                    ),
                    "filename_a": page1_filename[
                        anchor_writer
                    ],
                    "filename_b": page2_filename[
                        negative_writer
                    ],
                    "pair_label": 0,
                }
            )

        schedule_rows.append(
            pd.DataFrame(
                epoch_rows
            )
        )

    return pd.concat(
        schedule_rows,
        ignore_index=True,
    )


training_pairs_df = (
    build_dynamic_training_pairs()
)

print(
    "Writers:",
    len(
        writers
    ),
)

print(
    "Epoch offsets:",
    (
        training_pairs_df[
            [
                "epoch",
                "offset",
            ]
        ]
        .drop_duplicates()
        [
            "offset"
        ]
        .tolist()
    ),
)

print(
    "Pairs per epoch:",
    training_pairs_df.groupby(
        "epoch"
    ).size().unique().tolist(),
)

print(
    "Genuine pairs per epoch:",
    (
        training_pairs_df[
            training_pairs_df[
                "pair_label"
            ] == 1
        ]
        .groupby(
            "epoch"
        )
        .size()
        .unique()
        .tolist()
    ),
)

print(
    "Impostor pairs per epoch:",
    (
        training_pairs_df[
            training_pairs_df[
                "pair_label"
            ] == 0
        ]
        .groupby(
            "epoch"
        )
        .size()
        .unique()
        .tolist()
    ),
)

assert len(
    writers
) == 226

assert (
    training_pairs_df[
        [
            "epoch",
            "offset",
        ]
    ]
    .drop_duplicates()
    [
        "offset"
    ]
    .tolist()
) == DYNAMIC_NEGATIVE_OFFSETS

Writers: 226
Epoch offsets: [222, 168, 20, 143, 97, 96, 156, 22, 46, 190]
Pairs per epoch: [452]
Genuine pairs per epoch: [226]
Impostor pairs per epoch: [226]


In [12]:
class QUWIPairDataset(Dataset):
    def __init__(
        self,
        pair_df,
        image_dir,
    ):
        self.pair_df = (
            pair_df
            .reset_index(drop=True)
            .copy()
        )

        self.image_dir = Path(
            image_dir
        )

    def __len__(self):
        return len(
            self.pair_df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.pair_df.iloc[
            index
        ]

        image_a, _ = load_image_tensor(
            self.image_dir
            / row[
                "filename_a"
            ]
        )

        image_b, _ = load_image_tensor(
            self.image_dir
            / row[
                "filename_b"
            ]
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "pair_label": torch.tensor(
                row[
                    "pair_label"
                ],
                dtype=torch.float32,
            ),
        }


script_dataset = QUWIDataset(
    metadata=development_df,
    image_dir=IMAGE_DIR,
)


def create_metric_loader(
    epoch,
    training_seed,
):
    epoch_pairs_df = (
        training_pairs_df[
            training_pairs_df[
                "epoch"
            ] == epoch
        ]
        .reset_index(drop=True)
    )

    pair_dataset = QUWIPairDataset(
        epoch_pairs_df,
        IMAGE_DIR,
    )

    generator = (
        torch.Generator()
        .manual_seed(
            training_seed
            + epoch
        )
    )

    loader = DataLoader(
        pair_dataset,
        batch_size=METRIC_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        generator=generator,
    )

    return (
        epoch_pairs_df,
        loader,
    )


def create_script_loader(
    epoch,
    training_seed,
):
    generator = (
        torch.Generator()
        .manual_seed(
            training_seed
            + 10_000
            + epoch
        )
    )

    return DataLoader(
        script_dataset,
        batch_size=SCRIPT_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        generator=generator,
    )


epoch1_pairs_df, epoch1_metric_loader = (
    create_metric_loader(
        epoch=1,
        training_seed=42,
    )
)

epoch1_script_loader = (
    create_script_loader(
        epoch=1,
        training_seed=42,
    )
)

metric_batch = next(
    iter(
        epoch1_metric_loader
    )
)

script_batch = next(
    iter(
        epoch1_script_loader
    )
)

print(
    "Metric pairs per epoch:",
    len(
        epoch1_pairs_df
    ),
)

print(
    "Metric batches per epoch:",
    len(
        epoch1_metric_loader
    ),
)

print(
    "Script images per epoch:",
    len(
        script_dataset
    ),
)

print(
    "Script batches per epoch:",
    len(
        epoch1_script_loader
    ),
)

print(
    "Metric batch shape:",
    metric_batch[
        "image_a"
    ].shape,
)

print(
    "Script batch shape:",
    script_batch[
        "image"
    ].shape,
)

assert len(
    epoch1_pairs_df
) == 452

assert len(
    epoch1_metric_loader
) == 57

assert len(
    script_dataset
) == 904

assert len(
    epoch1_script_loader
) == 57

Metric pairs per epoch: 452
Metric batches per epoch: 57
Script images per epoch: 904
Script batches per epoch: 57
Metric batch shape: torch.Size([8, 3, 384, 384])
Script batch shape: torch.Size([16, 3, 384, 384])


In [9]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


class GradientReversalFunction(Function):
    @staticmethod
    def forward(
        ctx,
        input_tensor,
        coefficient,
    ):
        ctx.coefficient = coefficient

        return input_tensor.view_as(
            input_tensor
        )

    @staticmethod
    def backward(
        ctx,
        gradient_output,
    ):
        return (
            -ctx.coefficient
            * gradient_output,
            None,
        )


def gradient_reverse(
    input_tensor,
    coefficient,
):
    return GradientReversalFunction.apply(
        input_tensor,
        coefficient,
    )


class ScriptAdversarialWriterModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = resnet18(
            weights=None
        )

        self.backbone.fc = nn.Identity()

        self.script_classifier = nn.Linear(
            512,
            2,
        )

        self.register_buffer(
            "imagenet_mean",
            IMAGENET_MEAN,
        )

        self.register_buffer(
            "imagenet_std",
            IMAGENET_STD,
        )

        for parameter in (
            self.backbone.parameters()
        ):
            parameter.requires_grad = False

        for parameter in (
            self.backbone.layer4.parameters()
        ):
            parameter.requires_grad = True

    def train(
        self,
        mode=True,
    ):
        super().train(
            mode
        )

        self.backbone.eval()

        self.script_classifier.train(
            mode
        )

        return self

    def encode(
        self,
        images,
    ):
        normalized_images = (
            images
            - self.imagenet_mean
        ) / self.imagenet_std

        embeddings = self.backbone(
            normalized_images
        )

        return F.normalize(
            embeddings,
            p=2,
            dim=1,
        )

    def classify_script(
        self,
        embeddings,
        coefficient,
    ):
        reversed_embeddings = (
            gradient_reverse(
                embeddings,
                coefficient,
            )
        )

        return self.script_classifier(
            reversed_embeddings
        )


ssl_checkpoint = torch.load(
    SSL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

ssl_backbone_state = {
    key.replace(
        "encoder_q.backbone.",
        "",
        1,
    ): value
    for key, value in (
        ssl_checkpoint[
            "model_state_dict"
        ].items()
    )
    if key.startswith(
        "encoder_q.backbone."
    )
}


def build_fresh_adversarial_model(
    training_seed,
):
    set_training_seed(
        training_seed
    )

    model = (
        ScriptAdversarialWriterModel()
    )

    load_result = (
        model.backbone.load_state_dict(
            ssl_backbone_state,
            strict=True,
        )
    )

    model = model.to(
        TRAIN_DEVICE
    )

    return model


metric_criterion = nn.CosineEmbeddingLoss(
    margin=COSINE_MARGIN
)

script_criterion = nn.CrossEntropyLoss()


sanity_model = (
    build_fresh_adversarial_model(
        42
    )
)

sanity_images = script_batch[
    "image"
][
    :4
].to(
    TRAIN_DEVICE
)

with torch.no_grad():
    sanity_embeddings = (
        sanity_model.encode(
            sanity_images
        )
    )

print(
    "SSL checkpoint epoch:",
    ssl_checkpoint[
        "epoch"
    ],
)

print(
    "SSL backbone tensors:",
    len(
        ssl_backbone_state
    ),
)

print(
    "Embedding shape:",
    sanity_embeddings.shape,
)

print(
    "Embedding norms:",
    torch.linalg.vector_norm(
        sanity_embeddings,
        dim=1,
    )
    .detach()
    .cpu()
    .numpy(),
)

print(
    "Early backbone trainable:",
    any(
        parameter.requires_grad
        for parameter in (
            sanity_model
            .backbone
            .layer3
            .parameters()
        )
    ),
)

print(
    "Layer 4 trainable:",
    any(
        parameter.requires_grad
        for parameter in (
            sanity_model
            .backbone
            .layer4
            .parameters()
        )
    ),
)

del sanity_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

SSL checkpoint epoch: 20
SSL backbone tensors: 120
Embedding shape: torch.Size([4, 512])
Embedding norms: [1.         1.         1.         0.99999994]
Early backbone trainable: False
Layer 4 trainable: True


In [10]:
validation_dataset = QUWIDataset(
    metadata=validation_df,
    image_dir=IMAGE_DIR,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = thresholds[
            nearest_index
        ]

        return float(
            eer
        ), float(
            threshold
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return float(
        eer
    ), float(
        threshold
    )


def extract_embedding_map(
    model,
    loader,
):
    model.eval()

    embedding_map = {}

    with torch.no_grad():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            embeddings = (
                model.encode(
                    images
                )
                .detach()
                .cpu()
                .numpy()
            )

            for filename, embedding in zip(
                batch[
                    "filename"
                ],
                embeddings,
            ):
                embedding_map[
                    filename
                ] = embedding

    return embedding_map


def evaluate_validation_verification(
    model,
):
    embedding_map = (
        extract_embedding_map(
            model,
            validation_loader,
        )
    )

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    labels = validation_pairs_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, eer_threshold = (
        calculate_interpolated_eer(
            labels,
            scores,
        )
    )

    return {
        "auc": float(
            auc
        ),
        "eer": float(
            eer
        ),
        "eer_threshold": float(
            eer_threshold
        ),
    }


seed42_initial_model = (
    build_fresh_adversarial_model(
        42
    )
)

seed42_initial_metrics = (
    evaluate_validation_verification(
        seed42_initial_model
    )
)

print(
    "Validation images:",
    len(
        validation_dataset
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print(
    "Fresh SSL validation AUC:",
    seed42_initial_metrics[
        "auc"
    ],
)

print(
    "Fresh SSL validation EER (%):",
    100.0
    * seed42_initial_metrics[
        "eer"
    ],
)

del seed42_initial_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Validation images: 224
Validation pairs: 18816
Fresh SSL validation AUC: 0.6737687783446711
Fresh SSL validation EER (%): 38.988095238095234


In [13]:
def train_joint_epoch(
    model,
    epoch,
    training_seed,
    grl_coefficient,
    encoder_optimizer,
    script_optimizer,
):
    _, metric_loader = create_metric_loader(
        epoch=epoch,
        training_seed=training_seed,
    )

    script_loader = create_script_loader(
        epoch=epoch,
        training_seed=training_seed,
    )

    assert len(
        metric_loader
    ) == len(
        script_loader
    )

    model.train()

    encoder_parameters = [
        parameter
        for parameter in (
            model
            .backbone
            .layer4
            .parameters()
        )
        if parameter.requires_grad
    ]

    script_parameters = list(
        model
        .script_classifier
        .parameters()
    )

    metric_loss_sum = 0.0
    script_loss_sum = 0.0
    script_correct = 0
    script_total = 0
    encoder_grad_norm_sum = 0.0

    for (
        metric_batch,
        script_batch_current,
    ) in zip(
        metric_loader,
        script_loader,
    ):
        encoder_optimizer.zero_grad(
            set_to_none=True
        )

        script_optimizer.zero_grad(
            set_to_none=True
        )

        image_a = (
            metric_batch[
                "image_a"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        image_b = (
            metric_batch[
                "image_b"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        metric_targets = (
            metric_batch[
                "pair_label"
            ]
            .to(
                TRAIN_DEVICE
            )
            .float()
            .mul(
                2.0
            )
            .sub(
                1.0
            )
        )

        embedding_a = model.encode(
            image_a
        )

        embedding_b = model.encode(
            image_b
        )

        metric_loss = metric_criterion(
            embedding_a,
            embedding_b,
            metric_targets,
        )

        script_images = (
            script_batch_current[
                "image"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        script_labels = (
            script_batch_current[
                "language_label"
            ]
            .long()
            .to(
                TRAIN_DEVICE
            )
        )

        script_embeddings = model.encode(
            script_images
        )

        script_logits = (
            model.classify_script(
                script_embeddings,
                grl_coefficient,
            )
        )

        script_loss = script_criterion(
            script_logits,
            script_labels,
        )

        total_loss = (
            metric_loss
            + script_loss
        )

        total_loss.backward()

        encoder_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                encoder_parameters,
                max_norm=(
                    GRAD_CLIP_MAX_NORM
                ),
            )
        )

        torch.nn.utils.clip_grad_norm_(
            script_parameters,
            max_norm=(
                GRAD_CLIP_MAX_NORM
            ),
        )

        encoder_optimizer.step()
        script_optimizer.step()

        predictions = script_logits.argmax(
            dim=1
        )

        metric_loss_sum += float(
            metric_loss
            .detach()
            .cpu()
        )

        script_loss_sum += float(
            script_loss
            .detach()
            .cpu()
        )

        script_correct += int(
            (
                predictions
                == script_labels
            )
            .sum()
            .detach()
            .cpu()
        )

        script_total += int(
            script_labels.shape[
                0
            ]
        )

        encoder_grad_norm_sum += float(
            encoder_grad_norm
            .detach()
            .cpu()
        )

    batch_count = len(
        metric_loader
    )

    return {
        "metric_loss": (
            metric_loss_sum
            / batch_count
        ),
        "script_loss": (
            script_loss_sum
            / batch_count
        ),
        "script_accuracy": (
            script_correct
            / script_total
        ),
        "encoder_grad_norm": (
            encoder_grad_norm_sum
            / batch_count
        ),
    }


print(
    "Joint trainer ready."
)

Joint trainer ready.


In [14]:
def train_alternating_epoch(
    model,
    epoch,
    training_seed,
    grl_coefficient,
    encoder_optimizer,
    script_optimizer,
    adversary_steps,
):
    _, metric_loader = create_metric_loader(
        epoch=epoch,
        training_seed=training_seed,
    )

    script_loader = create_script_loader(
        epoch=epoch,
        training_seed=training_seed,
    )

    assert len(
        metric_loader
    ) == len(
        script_loader
    )

    model.train()

    metric_loss_sum = 0.0
    script_loss_sum = 0.0
    script_correct = 0
    script_total = 0
    encoder_grad_norm_sum = 0.0

    encoder_parameters = [
        parameter
        for parameter in (
            model
            .backbone
            .layer4
            .parameters()
        )
        if parameter.requires_grad
    ]

    for (
        metric_batch,
        script_batch_current,
    ) in zip(
        metric_loader,
        script_loader,
    ):
        script_images = (
            script_batch_current[
                "image"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        script_labels = (
            script_batch_current[
                "language_label"
            ]
            .long()
            .to(
                TRAIN_DEVICE
            )
        )

        with torch.no_grad():
            detached_script_embeddings = (
                model.encode(
                    script_images
                )
            )

        for _ in range(
            adversary_steps
        ):
            script_optimizer.zero_grad(
                set_to_none=True
            )

            script_logits = (
                model
                .script_classifier(
                    detached_script_embeddings
                )
            )

            script_loss = script_criterion(
                script_logits,
                script_labels,
            )

            script_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model
                .script_classifier
                .parameters(),
                max_norm=(
                    GRAD_CLIP_MAX_NORM
                ),
            )

            script_optimizer.step()

        for parameter in (
            model
            .script_classifier
            .parameters()
        ):
            parameter.requires_grad = False

        encoder_optimizer.zero_grad(
            set_to_none=True
        )

        image_a = (
            metric_batch[
                "image_a"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        image_b = (
            metric_batch[
                "image_b"
            ]
            .to(
                TRAIN_DEVICE
            )
        )

        metric_targets = (
            metric_batch[
                "pair_label"
            ]
            .to(
                TRAIN_DEVICE
            )
            .float()
            .mul(
                2.0
            )
            .sub(
                1.0
            )
        )

        embedding_a = model.encode(
            image_a
        )

        embedding_b = model.encode(
            image_b
        )

        metric_loss = metric_criterion(
            embedding_a,
            embedding_b,
            metric_targets,
        )

        script_embeddings = model.encode(
            script_images
        )

        adversarial_logits = (
            model.classify_script(
                script_embeddings,
                grl_coefficient,
            )
        )

        adversarial_script_loss = (
            script_criterion(
                adversarial_logits,
                script_labels,
            )
        )

        total_loss = (
            metric_loss
            + adversarial_script_loss
        )

        total_loss.backward()

        encoder_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                encoder_parameters,
                max_norm=(
                    GRAD_CLIP_MAX_NORM
                ),
            )
        )

        encoder_optimizer.step()

        for parameter in (
            model
            .script_classifier
            .parameters()
        ):
            parameter.requires_grad = True

        with torch.no_grad():
            final_script_logits = (
                model
                .script_classifier(
                    model.encode(
                        script_images
                    )
                )
            )

            final_predictions = (
                final_script_logits.argmax(
                    dim=1
                )
            )

        metric_loss_sum += float(
            metric_loss
            .detach()
            .cpu()
        )

        script_loss_sum += float(
            adversarial_script_loss
            .detach()
            .cpu()
        )

        script_correct += int(
            (
                final_predictions
                == script_labels
            )
            .sum()
            .detach()
            .cpu()
        )

        script_total += int(
            script_labels.shape[
                0
            ]
        )

        encoder_grad_norm_sum += float(
            encoder_grad_norm
            .detach()
            .cpu()
        )

    batch_count = len(
        metric_loader
    )

    return {
        "metric_loss": (
            metric_loss_sum
            / batch_count
        ),
        "script_loss": (
            script_loss_sum
            / batch_count
        ),
        "script_accuracy": (
            script_correct
            / script_total
        ),
        "encoder_grad_norm": (
            encoder_grad_norm_sum
            / batch_count
        ),
    }


print(
    "Batch-alternating trainer ready."
)

Batch-alternating trainer ready.


In [15]:
def run_control_experiment(
    training_seed,
):
    set_training_seed(
        training_seed
    )

    model = (
        build_fresh_adversarial_model(
            training_seed
        )
    )

    encoder_optimizer = torch.optim.AdamW(
        model
        .backbone
        .layer4
        .parameters(),
        lr=ENCODER_LR,
        weight_decay=WEIGHT_DECAY,
    )

    script_optimizer = torch.optim.AdamW(
        model
        .script_classifier
        .parameters(),
        lr=SCRIPT_HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"seed{training_seed}_lambda0_control_best.pt"
    )

    history = []

    best_auc = -np.inf
    best_eer = np.inf
    best_epoch = None

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        training_metrics = (
            train_joint_epoch(
                model=model,
                epoch=epoch,
                training_seed=training_seed,
                grl_coefficient=0.0,
                encoder_optimizer=(
                    encoder_optimizer
                ),
                script_optimizer=(
                    script_optimizer
                ),
            )
        )

        validation_metrics = (
            evaluate_validation_verification(
                model
            )
        )

        current_auc = (
            validation_metrics[
                "auc"
            ]
        )

        current_eer = (
            validation_metrics[
                "eer"
            ]
        )

        auc_improved = (
            current_auc
            > best_auc
        )

        auc_tied = np.isclose(
            current_auc,
            best_auc,
            rtol=0.0,
            atol=1e-12,
        )

        eer_improved_on_tie = (
            auc_tied
            and current_eer
            < best_eer
        )

        selected = (
            auc_improved
            or eer_improved_on_tie
        )

        if selected:
            best_auc = current_auc
            best_eer = current_eer
            best_epoch = epoch

            torch.save(
                {
                    "training_seed": (
                        training_seed
                    ),
                    "epoch": epoch,
                    "model": (
                        "lambda0_control"
                    ),
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "validation_auc": (
                        current_auc
                    ),
                    "validation_eer": (
                        current_eer
                    ),
                    "validation_eer_threshold": (
                        validation_metrics[
                            "eer_threshold"
                        ]
                    ),
                },
                checkpoint_path,
            )

        history.append(
            {
                "training_seed": (
                    training_seed
                ),
                "epoch": epoch,
                "metric_loss": (
                    training_metrics[
                        "metric_loss"
                    ]
                ),
                "script_loss": (
                    training_metrics[
                        "script_loss"
                    ]
                ),
                "script_accuracy": (
                    training_metrics[
                        "script_accuracy"
                    ]
                ),
                "encoder_grad_norm": (
                    training_metrics[
                        "encoder_grad_norm"
                    ]
                ),
                "validation_auc": (
                    current_auc
                ),
                "validation_eer": (
                    current_eer
                ),
                "checkpoint_selected": (
                    selected
                ),
            }
        )

        print(
            f"Seed {training_seed} | "
            f"Epoch {epoch:02d} | "
            f"Metric loss "
            f"{training_metrics['metric_loss']:.6f} | "
            f"Val AUC "
            f"{current_auc:.6f} | "
            f"Val EER "
            f"{100.0 * current_eer:.4f}%"
            + (
                " | selected"
                if selected
                else ""
            )
        )

    history_df = pd.DataFrame(
        history
    )

    print()

    print(
        "Best epoch:",
        best_epoch,
    )

    print(
        "Best validation AUC:",
        best_auc,
    )

    print(
        "Best validation EER (%):",
        100.0
        * best_eer,
    )

    print(
        "Checkpoint:",
        checkpoint_path,
    )

    return {
        "training_seed": (
            training_seed
        ),
        "best_epoch": (
            best_epoch
        ),
        "best_auc": (
            best_auc
        ),
        "best_eer": (
            best_eer
        ),
        "checkpoint_path": (
            checkpoint_path
        ),
        "history": (
            history_df
        ),
    }


print(
    "Control experiment runner ready."
)

Control experiment runner ready.


In [16]:
seed123_control_result = (
    run_control_experiment(
        training_seed=123
    )
)

Seed 123 | Epoch 01 | Metric loss 0.196130 | Val AUC 0.706834 | Val EER 35.7035% | selected
Seed 123 | Epoch 02 | Metric loss 0.178804 | Val AUC 0.713885 | Val EER 34.7727% | selected
Seed 123 | Epoch 03 | Metric loss 0.163152 | Val AUC 0.721626 | Val EER 34.5238% | selected
Seed 123 | Epoch 04 | Metric loss 0.150500 | Val AUC 0.730624 | Val EER 33.0519% | selected
Seed 123 | Epoch 05 | Metric loss 0.151573 | Val AUC 0.732862 | Val EER 33.0357% | selected
Seed 123 | Epoch 06 | Metric loss 0.139472 | Val AUC 0.738789 | Val EER 32.7381% | selected
Seed 123 | Epoch 07 | Metric loss 0.133319 | Val AUC 0.731650 | Val EER 34.2911%
Seed 123 | Epoch 08 | Metric loss 0.124931 | Val AUC 0.732787 | Val EER 33.9286%
Seed 123 | Epoch 09 | Metric loss 0.124657 | Val AUC 0.717462 | Val EER 34.5238%
Seed 123 | Epoch 10 | Metric loss 0.117234 | Val AUC 0.723404 | Val EER 35.0216%

Best epoch: 6
Best validation AUC: 0.7387890383426098
Best validation EER (%): 32.738095238095234
Checkpoint: /home/arijit/

In [17]:
def run_batch_alternating_experiment(
    training_seed,
):
    set_training_seed(
        training_seed
    )

    model = (
        build_fresh_adversarial_model(
            training_seed
        )
    )

    encoder_optimizer = torch.optim.AdamW(
        model
        .backbone
        .layer4
        .parameters(),
        lr=ENCODER_LR,
        weight_decay=WEIGHT_DECAY,
    )

    script_optimizer = torch.optim.AdamW(
        model
        .script_classifier
        .parameters(),
        lr=SCRIPT_HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"seed{training_seed}_batch_alt_lambda0p5_best.pt"
    )

    history = []

    best_auc = -np.inf
    best_eer = np.inf
    best_epoch = None

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        training_metrics = (
            train_alternating_epoch(
                model=model,
                epoch=epoch,
                training_seed=training_seed,
                grl_coefficient=GRL_COEFFICIENT,
                encoder_optimizer=(
                    encoder_optimizer
                ),
                script_optimizer=(
                    script_optimizer
                ),
                adversary_steps=(
                    ADVERSARY_STEPS_PER_BATCH
                ),
            )
        )

        validation_metrics = (
            evaluate_validation_verification(
                model
            )
        )

        current_auc = (
            validation_metrics[
                "auc"
            ]
        )

        current_eer = (
            validation_metrics[
                "eer"
            ]
        )

        auc_improved = (
            current_auc
            > best_auc
        )

        auc_tied = np.isclose(
            current_auc,
            best_auc,
            rtol=0.0,
            atol=1e-12,
        )

        eer_improved_on_tie = (
            auc_tied
            and current_eer
            < best_eer
        )

        selected = (
            auc_improved
            or eer_improved_on_tie
        )

        if selected:
            best_auc = current_auc
            best_eer = current_eer
            best_epoch = epoch

            torch.save(
                {
                    "training_seed": (
                        training_seed
                    ),
                    "epoch": epoch,
                    "model": (
                        "batch_alt_lambda0p5"
                    ),
                    "grl_coefficient": (
                        GRL_COEFFICIENT
                    ),
                    "adversary_steps_per_batch": (
                        ADVERSARY_STEPS_PER_BATCH
                    ),
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "validation_auc": (
                        current_auc
                    ),
                    "validation_eer": (
                        current_eer
                    ),
                    "validation_eer_threshold": (
                        validation_metrics[
                            "eer_threshold"
                        ]
                    ),
                },
                checkpoint_path,
            )

        history.append(
            {
                "training_seed": (
                    training_seed
                ),
                "epoch": epoch,
                "metric_loss": (
                    training_metrics[
                        "metric_loss"
                    ]
                ),
                "script_loss": (
                    training_metrics[
                        "script_loss"
                    ]
                ),
                "script_accuracy": (
                    training_metrics[
                        "script_accuracy"
                    ]
                ),
                "encoder_grad_norm": (
                    training_metrics[
                        "encoder_grad_norm"
                    ]
                ),
                "validation_auc": (
                    current_auc
                ),
                "validation_eer": (
                    current_eer
                ),
                "checkpoint_selected": (
                    selected
                ),
            }
        )

        print(
            f"Seed {training_seed} | "
            f"Epoch {epoch:02d} | "
            f"Metric loss "
            f"{training_metrics['metric_loss']:.6f} | "
            f"Script loss "
            f"{training_metrics['script_loss']:.6f} | "
            f"Script acc "
            f"{training_metrics['script_accuracy']:.4f} | "
            f"Val AUC "
            f"{current_auc:.6f} | "
            f"Val EER "
            f"{100.0 * current_eer:.4f}%"
            + (
                " | selected"
                if selected
                else ""
            )
        )

    history_df = pd.DataFrame(
        history
    )

    print()

    print(
        "Best epoch:",
        best_epoch,
    )

    print(
        "Best validation AUC:",
        best_auc,
    )

    print(
        "Best validation EER (%):",
        100.0
        * best_eer,
    )

    print(
        "Checkpoint:",
        checkpoint_path,
    )

    return {
        "training_seed": (
            training_seed
        ),
        "best_epoch": (
            best_epoch
        ),
        "best_auc": (
            best_auc
        ),
        "best_eer": (
            best_eer
        ),
        "checkpoint_path": (
            checkpoint_path
        ),
        "history": (
            history_df
        ),
    }


print(
    "Batch-alternating experiment runner ready."
)

Batch-alternating experiment runner ready.


In [18]:
seed123_batch_alt_result = (
    run_batch_alternating_experiment(
        training_seed=123
    )
)

Seed 123 | Epoch 01 | Metric loss 0.197102 | Script loss 0.602542 | Script acc 0.9015 | Val AUC 0.706014 | Val EER 36.6071% | selected
Seed 123 | Epoch 02 | Metric loss 0.188405 | Script loss 0.641575 | Script acc 0.7467 | Val AUC 0.725775 | Val EER 34.2262% | selected
Seed 123 | Epoch 03 | Metric loss 0.170271 | Script loss 0.705870 | Script acc 0.4746 | Val AUC 0.741060 | Val EER 32.7381% | selected
Seed 123 | Epoch 04 | Metric loss 0.159898 | Script loss 0.698447 | Script acc 0.4867 | Val AUC 0.748762 | Val EER 32.3052% | selected
Seed 123 | Epoch 05 | Metric loss 0.159989 | Script loss 0.686979 | Script acc 0.5442 | Val AUC 0.756917 | Val EER 32.1104% | selected
Seed 123 | Epoch 06 | Metric loss 0.149342 | Script loss 0.699760 | Script acc 0.4591 | Val AUC 0.765500 | Val EER 30.9524% | selected
Seed 123 | Epoch 07 | Metric loss 0.141662 | Script loss 0.682810 | Script acc 0.5619 | Val AUC 0.748229 | Val EER 32.4405%
Seed 123 | Epoch 08 | Metric loss 0.137527 | Script loss 0.694955 

In [19]:
seed123_auc_gain = (
    seed123_batch_alt_result[
        "best_auc"
    ]
    - seed123_control_result[
        "best_auc"
    ]
)

seed123_eer_change_pp = (
    100.0
    * (
        seed123_batch_alt_result[
            "best_eer"
        ]
        - seed123_control_result[
            "best_eer"
        ]
    )
)

print(
    "Seed-123 control AUC:",
    seed123_control_result[
        "best_auc"
    ],
)

print(
    "Seed-123 batch-alt AUC:",
    seed123_batch_alt_result[
        "best_auc"
    ],
)

print(
    "Seed-123 AUC gain:",
    seed123_auc_gain,
)

print()

print(
    "Seed-123 control EER (%):",
    100.0
    * seed123_control_result[
        "best_eer"
    ],
)

print(
    "Seed-123 batch-alt EER (%):",
    100.0
    * seed123_batch_alt_result[
        "best_eer"
    ],
)

print(
    "Seed-123 EER change (pp):",
    seed123_eer_change_pp,
)

Seed-123 control AUC: 0.7387890383426098
Seed-123 batch-alt AUC: 0.7764246418264277
Seed-123 AUC gain: 0.03763560348381789

Seed-123 control EER (%): 32.738095238095234
Seed-123 batch-alt EER (%): 29.166666666666664
Seed-123 EER change (pp): -3.57142857142857


In [20]:
seed2026_control_result = (
    run_control_experiment(
        training_seed=2026
    )
)

Seed 2026 | Epoch 01 | Metric loss 0.196539 | Val AUC 0.704606 | Val EER 35.9578% | selected
Seed 2026 | Epoch 02 | Metric loss 0.177411 | Val AUC 0.720801 | Val EER 33.9286% | selected
Seed 2026 | Epoch 03 | Metric loss 0.163843 | Val AUC 0.725122 | Val EER 34.2262% | selected
Seed 2026 | Epoch 04 | Metric loss 0.152450 | Val AUC 0.733069 | Val EER 32.6732% | selected
Seed 2026 | Epoch 05 | Metric loss 0.152645 | Val AUC 0.732147 | Val EER 33.6310%
Seed 2026 | Epoch 06 | Metric loss 0.139946 | Val AUC 0.732816 | Val EER 33.9665%
Seed 2026 | Epoch 07 | Metric loss 0.132303 | Val AUC 0.738279 | Val EER 34.5238% | selected
Seed 2026 | Epoch 08 | Metric loss 0.124794 | Val AUC 0.736761 | Val EER 33.7229%
Seed 2026 | Epoch 09 | Metric loss 0.123676 | Val AUC 0.723044 | Val EER 34.6753%
Seed 2026 | Epoch 10 | Metric loss 0.115613 | Val AUC 0.727243 | Val EER 34.3994%

Best epoch: 7
Best validation AUC: 0.7382791563595135
Best validation EER (%): 34.523809523809526
Checkpoint: /home/arijit/D

In [21]:
seed2026_batch_alt_result = (
    run_batch_alternating_experiment(
        training_seed=2026
    )
)

Seed 2026 | Epoch 01 | Metric loss 0.199228 | Script loss 0.617446 | Script acc 0.8496 | Val AUC 0.706769 | Val EER 36.0119% | selected
Seed 2026 | Epoch 02 | Metric loss 0.187496 | Script loss 0.638333 | Script acc 0.6670 | Val AUC 0.729032 | Val EER 33.9286% | selected
Seed 2026 | Epoch 03 | Metric loss 0.172040 | Script loss 0.704068 | Script acc 0.4447 | Val AUC 0.743361 | Val EER 33.0357% | selected
Seed 2026 | Epoch 04 | Metric loss 0.161154 | Script loss 0.696349 | Script acc 0.5166 | Val AUC 0.743250 | Val EER 32.9708%
Seed 2026 | Epoch 05 | Metric loss 0.158974 | Script loss 0.686796 | Script acc 0.5365 | Val AUC 0.755610 | Val EER 31.2500% | selected
Seed 2026 | Epoch 06 | Metric loss 0.149410 | Script loss 0.706538 | Script acc 0.3628 | Val AUC 0.770031 | Val EER 30.0325% | selected
Seed 2026 | Epoch 07 | Metric loss 0.140721 | Script loss 0.689134 | Script acc 0.5177 | Val AUC 0.764631 | Val EER 31.2500%
Seed 2026 | Epoch 08 | Metric loss 0.135804 | Script loss 0.679747 | S

In [22]:
seed2026_auc_gain = (
    seed2026_batch_alt_result[
        "best_auc"
    ]
    - seed2026_control_result[
        "best_auc"
    ]
)

seed2026_eer_change_pp = (
    100.0
    * (
        seed2026_batch_alt_result[
            "best_eer"
        ]
        - seed2026_control_result[
            "best_eer"
        ]
    )
)

print(
    "Seed-2026 control AUC:",
    seed2026_control_result[
        "best_auc"
    ],
)

print(
    "Seed-2026 batch-alt AUC:",
    seed2026_batch_alt_result[
        "best_auc"
    ],
)

print(
    "Seed-2026 AUC gain:",
    seed2026_auc_gain,
)

print()

print(
    "Seed-2026 control EER (%):",
    100.0
    * seed2026_control_result[
        "best_eer"
    ],
)

print(
    "Seed-2026 batch-alt EER (%):",
    100.0
    * seed2026_batch_alt_result[
        "best_eer"
    ],
)

print(
    "Seed-2026 EER change (pp):",
    seed2026_eer_change_pp,
)

Seed-2026 control AUC: 0.7382791563595135
Seed-2026 batch-alt AUC: 0.7700305832560297
Seed-2026 AUC gain: 0.03175142689651622

Seed-2026 control EER (%): 34.523809523809526
Seed-2026 batch-alt EER (%): 30.032467532467532
Seed-2026 EER change (pp): -4.49134199134199


In [23]:
multiseed_writer_df = pd.DataFrame(
    [
        {
            "training_seed": 42,
            "control_auc": float(
                seed42_control_row[
                    "writer_auc"
                ]
            ),
            "batch_alt_auc": float(
                seed42_batch_alt_row[
                    "writer_auc"
                ]
            ),
            "control_eer": float(
                seed42_control_row[
                    "writer_eer"
                ]
            ),
            "batch_alt_eer": float(
                seed42_batch_alt_row[
                    "writer_eer"
                ]
            ),
        },
        {
            "training_seed": 123,
            "control_auc": (
                seed123_control_result[
                    "best_auc"
                ]
            ),
            "batch_alt_auc": (
                seed123_batch_alt_result[
                    "best_auc"
                ]
            ),
            "control_eer": (
                seed123_control_result[
                    "best_eer"
                ]
            ),
            "batch_alt_eer": (
                seed123_batch_alt_result[
                    "best_eer"
                ]
            ),
        },
        {
            "training_seed": 2026,
            "control_auc": (
                seed2026_control_result[
                    "best_auc"
                ]
            ),
            "batch_alt_auc": (
                seed2026_batch_alt_result[
                    "best_auc"
                ]
            ),
            "control_eer": (
                seed2026_control_result[
                    "best_eer"
                ]
            ),
            "batch_alt_eer": (
                seed2026_batch_alt_result[
                    "best_eer"
                ]
            ),
        },
    ]
)

multiseed_writer_df[
    "auc_gain"
] = (
    multiseed_writer_df[
        "batch_alt_auc"
    ]
    - multiseed_writer_df[
        "control_auc"
    ]
)

multiseed_writer_df[
    "eer_change_pp"
] = (
    100.0
    * (
        multiseed_writer_df[
            "batch_alt_eer"
        ]
        - multiseed_writer_df[
            "control_eer"
        ]
    )
)

print(
    multiseed_writer_df[
        [
            "training_seed",
            "control_auc",
            "batch_alt_auc",
            "auc_gain",
            "control_eer",
            "batch_alt_eer",
            "eer_change_pp",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Control AUC mean:",
    multiseed_writer_df[
        "control_auc"
    ].mean(),
)

print(
    "Control AUC std:",
    multiseed_writer_df[
        "control_auc"
    ].std(
        ddof=1
    ),
)

print(
    "Batch-alt AUC mean:",
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean(),
)

print(
    "Batch-alt AUC std:",
    multiseed_writer_df[
        "batch_alt_auc"
    ].std(
        ddof=1
    ),
)

print()

print(
    "Mean paired AUC gain:",
    multiseed_writer_df[
        "auc_gain"
    ].mean(),
)

print(
    "Paired AUC gain std:",
    multiseed_writer_df[
        "auc_gain"
    ].std(
        ddof=1
    ),
)

print(
    "Seeds with positive AUC gain:",
    int(
        (
            multiseed_writer_df[
                "auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        multiseed_writer_df
    ),
)

print()

print(
    "Mean EER change (pp):",
    multiseed_writer_df[
        "eer_change_pp"
    ].mean(),
)

print(
    "Seeds with lower EER:",
    int(
        (
            multiseed_writer_df[
                "eer_change_pp"
            ] < 0
        ).sum()
    ),
    "/",
    len(
        multiseed_writer_df
    ),
)

 training_seed  control_auc  batch_alt_auc  auc_gain  control_eer  batch_alt_eer  eer_change_pp
            42     0.744147       0.785574  0.041427     0.329978       0.286255      -4.372294
           123     0.738789       0.776425  0.037636     0.327381       0.291667      -3.571429
          2026     0.738279       0.770031  0.031751     0.345238       0.300325      -4.491342

Control AUC mean: 0.7404050603827391
Control AUC std: 0.0032506157734929933
Batch-alt AUC mean: 0.7773429715952037
Batch-alt AUC std: 0.00781214033923

Mean paired AUC gain: 0.03693791121246481
Paired AUC gain std: 0.004875225537588518
Seeds with positive AUC gain: 3 / 3

Mean EER change (pp): -4.145021645021646
Seeds with lower EER: 3 / 3


In [24]:
development_probe_loader = DataLoader(
    script_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_probe_arrays(
    model,
    loader,
):
    model.eval()

    embeddings = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            batch_embeddings = (
                model.encode(
                    images
                )
                .detach()
                .cpu()
                .numpy()
            )

            batch_labels = (
                batch[
                    "language_label"
                ]
                .detach()
                .cpu()
                .numpy()
            )

            embeddings.append(
                batch_embeddings
            )

            labels.append(
                batch_labels
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def evaluate_checkpoint_script_probe(
    checkpoint_path,
    training_seed,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = (
        build_fresh_adversarial_model(
            training_seed
        )
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    development_embeddings, development_labels = (
        extract_probe_arrays(
            model,
            development_probe_loader,
        )
    )

    validation_embeddings, validation_labels = (
        extract_probe_arrays(
            model,
            validation_loader,
        )
    )

    probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SPLIT_SEED,
                ),
            ),
        ]
    )

    probe.fit(
        development_embeddings,
        development_labels,
    )

    validation_probabilities = (
        probe.predict_proba(
            validation_embeddings
        )[
            :,
            1,
        ]
    )

    validation_predictions = (
        probe.predict(
            validation_embeddings
        )
    )

    accuracy = accuracy_score(
        validation_labels,
        validation_predictions,
    )

    auc = roc_auc_score(
        validation_labels,
        validation_probabilities,
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "accuracy": float(
            accuracy
        ),
        "auc": float(
            auc
        ),
    }


seed123_script_probe = (
    evaluate_checkpoint_script_probe(
        seed123_batch_alt_result[
            "checkpoint_path"
        ],
        training_seed=123,
    )
)

seed2026_script_probe = (
    evaluate_checkpoint_script_probe(
        seed2026_batch_alt_result[
            "checkpoint_path"
        ],
        training_seed=2026,
    )
)

print(
    "Seed-42 script probe AUC:",
    seed42_batch_alt_row[
        "script_probe_auc"
    ],
)

print(
    "Seed-123 script probe accuracy:",
    seed123_script_probe[
        "accuracy"
    ],
)

print(
    "Seed-123 script probe AUC:",
    seed123_script_probe[
        "auc"
    ],
)

print(
    "Seed-2026 script probe accuracy:",
    seed2026_script_probe[
        "accuracy"
    ],
)

print(
    "Seed-2026 script probe AUC:",
    seed2026_script_probe[
        "auc"
    ],
)

Seed-42 script probe AUC: 0.9987244897959184
Seed-123 script probe accuracy: 0.9955357142857143
Seed-123 script probe AUC: 0.9992028061224489
Seed-2026 script probe accuracy: 0.9910714285714286
Seed-2026 script probe AUC: 0.9996811224489797


In [25]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


def evaluate_condition_families(
    checkpoint_path,
    training_seed,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = (
        build_fresh_adversarial_model(
            training_seed
        )
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    embedding_map = (
        extract_embedding_map(
            model,
            validation_loader,
        )
    )

    evaluation_df = (
        validation_pairs_df
        .copy()
        .reset_index(
            drop=True
        )
    )

    evaluation_df[
        "page_a"
    ] = evaluation_df[
        "filename_a"
    ].map(
        filename_page_id
    )

    evaluation_df[
        "page_b"
    ] = evaluation_df[
        "filename_b"
    ].map(
        filename_page_id
    )

    evaluation_df[
        "condition"
    ] = [
        CONDITION_BY_PAGE_PAIR[
            (
                int(
                    page_a
                ),
                int(
                    page_b
                ),
            )
        ]
        for page_a, page_b in zip(
            evaluation_df[
                "page_a"
            ],
            evaluation_df[
                "page_b"
            ],
        )
    ]

    embeddings_a = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                evaluation_df[
                    "filename_a"
                ]
            )
        ]
    )

    embeddings_b = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                evaluation_df[
                    "filename_b"
                ]
            )
        ]
    )

    evaluation_df[
        "score"
    ] = np.sum(
        embeddings_a
        * embeddings_b,
        axis=1,
    )

    rows = []

    for condition in (
        WITHIN_SCRIPT_CONDITIONS
        + CROSS_SCRIPT_CONDITIONS
    ):
        condition_df = (
            evaluation_df[
                evaluation_df[
                    "condition"
                ] == condition
            ]
        )

        labels = condition_df[
            "pair_label"
        ].to_numpy(
            dtype=np.int64
        )

        scores = condition_df[
            "score"
        ].to_numpy()

        rows.append(
            {
                "condition": (
                    condition
                ),
                "auc": float(
                    roc_auc_score(
                        labels,
                        scores,
                    )
                ),
            }
        )

    condition_df = pd.DataFrame(
        rows
    )

    within_auc = float(
        condition_df[
            condition_df[
                "condition"
            ].isin(
                WITHIN_SCRIPT_CONDITIONS
            )
        ][
            "auc"
        ].mean()
    )

    cross_auc = float(
        condition_df[
            condition_df[
                "condition"
            ].isin(
                CROSS_SCRIPT_CONDITIONS
            )
        ][
            "auc"
        ].mean()
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "condition_metrics": (
            condition_df
        ),
        "within_macro_auc": (
            within_auc
        ),
        "cross_macro_auc": (
            cross_auc
        ),
        "within_cross_gap": (
            within_auc
            - cross_auc
        ),
    }


print(
    "Condition-family evaluator ready."
)

Condition-family evaluator ready.


In [26]:
seed123_control_family = (
    evaluate_condition_families(
        seed123_control_result[
            "checkpoint_path"
        ],
        training_seed=123,
    )
)

seed123_batch_family = (
    evaluate_condition_families(
        seed123_batch_alt_result[
            "checkpoint_path"
        ],
        training_seed=123,
    )
)

seed2026_control_family = (
    evaluate_condition_families(
        seed2026_control_result[
            "checkpoint_path"
        ],
        training_seed=2026,
    )
)

seed2026_batch_family = (
    evaluate_condition_families(
        seed2026_batch_alt_result[
            "checkpoint_path"
        ],
        training_seed=2026,
    )
)


multiseed_cross_df = pd.DataFrame(
    [
        {
            "training_seed": 42,
            "control_within_auc": float(
                seed42_control_row[
                    "within_macro_auc"
                ]
            ),
            "batch_within_auc": float(
                seed42_batch_alt_row[
                    "within_macro_auc"
                ]
            ),
            "control_cross_auc": float(
                seed42_control_row[
                    "cross_macro_auc"
                ]
            ),
            "batch_cross_auc": float(
                seed42_batch_alt_row[
                    "cross_macro_auc"
                ]
            ),
        },
        {
            "training_seed": 123,
            "control_within_auc": (
                seed123_control_family[
                    "within_macro_auc"
                ]
            ),
            "batch_within_auc": (
                seed123_batch_family[
                    "within_macro_auc"
                ]
            ),
            "control_cross_auc": (
                seed123_control_family[
                    "cross_macro_auc"
                ]
            ),
            "batch_cross_auc": (
                seed123_batch_family[
                    "cross_macro_auc"
                ]
            ),
        },
        {
            "training_seed": 2026,
            "control_within_auc": (
                seed2026_control_family[
                    "within_macro_auc"
                ]
            ),
            "batch_within_auc": (
                seed2026_batch_family[
                    "within_macro_auc"
                ]
            ),
            "control_cross_auc": (
                seed2026_control_family[
                    "cross_macro_auc"
                ]
            ),
            "batch_cross_auc": (
                seed2026_batch_family[
                    "cross_macro_auc"
                ]
            ),
        },
    ]
)

multiseed_cross_df[
    "within_auc_gain"
] = (
    multiseed_cross_df[
        "batch_within_auc"
    ]
    - multiseed_cross_df[
        "control_within_auc"
    ]
)

multiseed_cross_df[
    "cross_auc_gain"
] = (
    multiseed_cross_df[
        "batch_cross_auc"
    ]
    - multiseed_cross_df[
        "control_cross_auc"
    ]
)

print(
    multiseed_cross_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Mean cross-script AUC gain:",
    multiseed_cross_df[
        "cross_auc_gain"
    ].mean(),
)

print(
    "Cross-script gain std:",
    multiseed_cross_df[
        "cross_auc_gain"
    ].std(
        ddof=1
    ),
)

print(
    "Seeds with positive cross-script gain:",
    int(
        (
            multiseed_cross_df[
                "cross_auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        multiseed_cross_df
    ),
)

print()

print(
    "Mean within-script AUC gain:",
    multiseed_cross_df[
        "within_auc_gain"
    ].mean(),
)

 training_seed  control_within_auc  batch_within_auc  control_cross_auc  batch_cross_auc  within_auc_gain  cross_auc_gain
            42            0.852400          0.848582           0.727728         0.755593        -0.003818        0.027866
           123            0.847582          0.844057           0.723256         0.747536        -0.003525        0.024280
          2026            0.843025          0.845263           0.722940         0.735129         0.002238        0.012188

Mean cross-script AUC gain: 0.021444515306122458
Cross-script gain std: 0.008214127161978902
Seeds with positive cross-script gain: 3 / 3

Mean within-script AUC gain: -0.0017016465677179848


In [27]:
multiseed_writer_path = (
    REPORT_DIR
    / "multiseed_writer_summary.csv"
)

multiseed_cross_path = (
    REPORT_DIR
    / "multiseed_cross_script_summary.csv"
)

seed123_control_history_path = (
    REPORT_DIR
    / "seed123_control_history.csv"
)

seed123_batch_history_path = (
    REPORT_DIR
    / "seed123_batch_alt_history.csv"
)

seed2026_control_history_path = (
    REPORT_DIR
    / "seed2026_control_history.csv"
)

seed2026_batch_history_path = (
    REPORT_DIR
    / "seed2026_batch_alt_history.csv"
)


multiseed_writer_df.to_csv(
    multiseed_writer_path,
    index=False,
)

multiseed_cross_df.to_csv(
    multiseed_cross_path,
    index=False,
)

seed123_control_result[
    "history"
].to_csv(
    seed123_control_history_path,
    index=False,
)

seed123_batch_alt_result[
    "history"
].to_csv(
    seed123_batch_history_path,
    index=False,
)

seed2026_control_result[
    "history"
].to_csv(
    seed2026_control_history_path,
    index=False,
)

seed2026_batch_alt_result[
    "history"
].to_csv(
    seed2026_batch_history_path,
    index=False,
)


saved_paths = [
    multiseed_writer_path,
    multiseed_cross_path,
    seed123_control_history_path,
    seed123_batch_history_path,
    seed2026_control_history_path,
    seed2026_batch_history_path,
]


print(
    "Saved reports:"
)

for path in saved_paths:
    print(
        "-",
        path.relative_to(
            PROJECT_ROOT
        ),
    )


assert all(
    path.exists()
    for path in saved_paths
)

Saved reports:
- reports/multiseed_adversarial_robustness/multiseed_writer_summary.csv
- reports/multiseed_adversarial_robustness/multiseed_cross_script_summary.csv
- reports/multiseed_adversarial_robustness/seed123_control_history.csv
- reports/multiseed_adversarial_robustness/seed123_batch_alt_history.csv
- reports/multiseed_adversarial_robustness/seed2026_control_history.csv
- reports/multiseed_adversarial_robustness/seed2026_batch_alt_history.csv


In [28]:
multiseed_script_probe_auc = [
    float(
        seed42_batch_alt_row[
            "script_probe_auc"
        ]
    ),
    float(
        seed123_script_probe[
            "auc"
        ]
    ),
    float(
        seed2026_script_probe[
            "auc"
        ]
    ),
]


notebook17_summary = {
    "experiment": (
        "Multi-Seed Robustness of "
        "Script-Adversarial Writer Verification"
    ),
    "training_seeds": [
        42,
        123,
        2026,
    ],
    "models": [
        "lambda0_control",
        "batch_alt_lambda0p5",
    ],
    "writer_auc": {
        "control_mean": float(
            multiseed_writer_df[
                "control_auc"
            ].mean()
        ),
        "control_std": float(
            multiseed_writer_df[
                "control_auc"
            ].std(
                ddof=1
            )
        ),
        "batch_alt_mean": float(
            multiseed_writer_df[
                "batch_alt_auc"
            ].mean()
        ),
        "batch_alt_std": float(
            multiseed_writer_df[
                "batch_alt_auc"
            ].std(
                ddof=1
            )
        ),
        "mean_paired_gain": float(
            multiseed_writer_df[
                "auc_gain"
            ].mean()
        ),
        "paired_gain_std": float(
            multiseed_writer_df[
                "auc_gain"
            ].std(
                ddof=1
            )
        ),
        "positive_gain_seeds": int(
            (
                multiseed_writer_df[
                    "auc_gain"
                ] > 0
            ).sum()
        ),
    },
    "eer": {
        "mean_change_percentage_points": float(
            multiseed_writer_df[
                "eer_change_pp"
            ].mean()
        ),
        "improved_seeds": int(
            (
                multiseed_writer_df[
                    "eer_change_pp"
                ] < 0
            ).sum()
        ),
    },
    "cross_script_auc": {
        "mean_gain": float(
            multiseed_cross_df[
                "cross_auc_gain"
            ].mean()
        ),
        "gain_std": float(
            multiseed_cross_df[
                "cross_auc_gain"
            ].std(
                ddof=1
            )
        ),
        "positive_gain_seeds": int(
            (
                multiseed_cross_df[
                    "cross_auc_gain"
                ] > 0
            ).sum()
        ),
    },
    "within_script_auc": {
        "mean_gain": float(
            multiseed_cross_df[
                "within_auc_gain"
            ].mean()
        ),
    },
    "script_probe_auc": {
        "seed42": (
            multiseed_script_probe_auc[
                0
            ]
        ),
        "seed123": (
            multiseed_script_probe_auc[
                1
            ]
        ),
        "seed2026": (
            multiseed_script_probe_auc[
                2
            ]
        ),
        "mean": float(
            np.mean(
                multiseed_script_probe_auc
            )
        ),
    },
    "main_interpretation": (
        "Across all three evaluated training seeds, "
        "batch-level alternating adversarial training "
        "improved validation writer ROC-AUC, reduced EER, "
        "and improved cross-script macro ROC-AUC relative "
        "to the matched lambda-0 control. However, script "
        "remained almost perfectly linearly accessible "
        "from the learned embeddings."
    ),
    "unsupported_claim": (
        "These results do not establish "
        "script-invariant representations."
    ),
    "statistical_limitation": (
        "Three seeds provide an initial robustness check "
        "but are insufficient for a strong formal "
        "statistical-significance claim."
    ),
}


summary_path = (
    REPORT_DIR
    / "notebook17_multiseed_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook17_summary,
        file,
        indent=2,
    )


print(
    "Summary saved:",
    summary_path.relative_to(
        PROJECT_ROOT
    ),
)

print()

print(
    "Mean paired writer AUC gain:",
    multiseed_writer_df[
        "auc_gain"
    ].mean(),
)

print(
    "Mean cross-script AUC gain:",
    multiseed_cross_df[
        "cross_auc_gain"
    ].mean(),
)

print(
    "Mean script probe AUC:",
    np.mean(
        multiseed_script_probe_auc
    ),
)

Summary saved: reports/multiseed_adversarial_robustness/notebook17_multiseed_summary.json

Mean paired writer AUC gain: 0.03693791121246481
Mean cross-script AUC gain: 0.021444515306122458
Mean script probe AUC: 0.999202806122449


## Final Summary

This notebook tested whether the validation improvement observed for batch-level alternating script-adversarial writer metric learning was robust to training seed.

The experiment compared the matched lambda-0 control against batch-level alternating adversarial training with GRL coefficient 0.5 and three adversary updates per metric batch. The writer split, SSL initialization, dynamic negative schedule, preprocessing, verification protocol, model architecture, optimizer settings, training duration, and AUC-first checkpoint-selection rule were held fixed. Only training stochasticity was varied across seeds 42, 123, and 2026.

### Multi-seed writer-verification results

| Seed | Control AUC | Batch-alt AUC | AUC gain | Control EER | Batch-alt EER | EER change |
|---|---:|---:|---:|---:|---:|---:|
| 42 | 0.744147 | 0.785574 | +0.041427 | 32.9978% | 28.6255% | -4.3723 pp |
| 123 | 0.738789 | 0.776425 | +0.037636 | 32.7381% | 29.1667% | -3.5714 pp |
| 2026 | 0.738279 | 0.770031 | +0.031751 | 34.5238% | 30.0325% | -4.4913 pp |

Across the three seeds:

- Control writer AUC: 0.740405 ± 0.003251
- Batch-alternating writer AUC: 0.777343 ± 0.007812
- Mean paired writer AUC gain: +0.036938 ± 0.004875
- Positive writer AUC gain: 3/3 seeds
- Mean EER change: -4.1450 percentage points
- Lower EER: 3/3 seeds

The writer-verification improvement therefore reproduced across all three evaluated training seeds rather than appearing only for the original seed-42 run.

### Cross-script robustness

Cross-script macro AUC also improved for every evaluated seed.

| Seed | Control cross-script AUC | Batch-alt cross-script AUC | Gain |
|---|---:|---:|---:|
| 42 | 0.727728 | 0.755593 | +0.027866 |
| 123 | 0.723256 | 0.747536 | +0.024280 |
| 2026 | 0.722940 | 0.735129 | +0.012188 |

The mean cross-script macro-AUC gain was +0.021445 ± 0.008214, with positive gains in 3/3 seeds.

In contrast, the mean within-script macro-AUC change was -0.001702. This suggests that the overall verification improvement was concentrated primarily in cross-script comparisons rather than reflecting a uniform improvement across all verification conditions.

### Script-accessibility result

Independent writer-disjoint linear script probes remained extremely strong:

- Seed 42 batch-alt script-probe AUC: 0.998724
- Seed 123 batch-alt script-probe AUC: 0.999203
- Seed 2026 batch-alt script-probe AUC: 0.999681
- Mean script-probe AUC: 0.999203

Therefore, the verification improvement was again observed without meaningful suppression of linearly accessible script information.

### Interpretation

The main Notebook 16 finding is robust to the three evaluated training seeds: batch-level alternating adversarial training consistently improves validation writer verification and cross-script verification relative to the matched lambda-0 control.

However, the independent script probes show that script information remains almost perfectly accessible from the learned embeddings. The improvement therefore cannot be interpreted as evidence that the representation became script-invariant.

The results support a distinction between adversarial optimization behavior and actual nuisance-information removal: improved cross-script verification and a confused attached adversarial head do not by themselves demonstrate script invariance.

### Limitations

This is an initial three-seed robustness study. Three seeds are useful for checking whether the observed effect is obviously seed-specific, but they are not sufficient for a strong formal statistical-significance claim.

The official test set was not used for checkpoint selection, adversarial-strength selection, or multi-seed model selection in this notebook. Stronger claims will require additional robustness analysis and evaluation on untouched external data.